# 03 - Predator-Prey: when your opponent keeps changing

**Setting: mixed cooperative-competitive.** 3 slower predators (red) chase 1 faster prey (green) around 2 obstacles.
* **Predators** share one team reward: every step, **+10 for each predator touching the prey** (all predators get it).
* **Prey**: -10 for each predator touching it, plus a penalty for leaving the arena.

We count **touch-steps per episode**: one predator touching the prey for one step = 1 (the episode has 25 steps; the prey is not removed when touched).

**Training schedule:** for the first 0.5M steps the prey does not learn. Its untrained network outputs "push equally in all
directions", so it stands still. Then it starts learning. Both teams use the same algorithm in each run (IDDPG or MADDPG).

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from marl.analysis import *          # load_run, describe_networks, evolution_frames, plot_curves, ...
from marl.render import show_video
plt.rcParams["figure.dpi"] = 90

## The Markov game

4 agents in two teams - a **mixed** (cooperative within a team, competitive between teams) Markov game.
One episode = **25 steps**; all agents act at the same time; nobody is removed when touched.

| | each of the 3 predators (red) | the prey (green) |
|---|---|---|
| **body** | bigger, slower (max speed 1.0) | smaller, faster (max speed 1.3, stronger push) |
| **observation** | 16 numbers: own velocity (2), own position (2), the 2 obstacles relative to itself (2 x 2), the 3 other agents relative to itself (3 x 2), **the prey's velocity** (2) | 14 numbers: own velocity (2), own position (2), the 2 obstacles relative to itself (2 x 2), the 3 predators relative to itself (3 x 2) |
| **action** | 5 numbers in [0, 1] = [no-op, push left, push right, push down, push up] | same |
| **reward (every step)** | **+10 for every predator touching the prey** - the same number for all 3 predators (team reward) | **-10 for every predator touching it**, minus a penalty for leaving the arena (0 inside 0.9, then growing quickly beyond the edge) |

Two big grey **obstacles** block movement for everybody. Everybody sees everything (full observability) - the difficulty is not
hidden information but the other team, whose strategy keeps changing as it learns.

The spaces, read from the environment, and a game between **untrained (random) players**:

In [ ]:
from marl.render import make_mpe, describe, rollout
env = make_mpe("tag")
describe(env)
frames, returns = rollout(env, seed=1, scale=0.6, title="random players")      # no policy = random actions
env.close()
print("returns of this episode:", {a: round(float(r), 1) for a, r in returns[0].items()})
show_video(frames, fps=10)

In [ ]:
iddpg, maddpg = load_run("tag_iddpg"), load_run("tag_maddpg")
freeze = maddpg["config"]["freeze"]["prey"]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
plot_curves([iddpg, maddpg], "contacts", ax=ax[0], title="touch-steps per episode (predators' success)",
            ylabel="touch-steps", vlines=[(freeze, "prey starts learning")])
plot_curves([iddpg, maddpg], "prey_min_dist", ax=ax[1], title="distance from the prey to its closest predator",
            ylabel="distance", vlines=[(freeze, "prey starts learning")])
plt.tight_layout()

As long as the prey stands still, the predators get better and better. As soon as the prey starts learning, their success
**collapses** - nothing changed for the predators except their opponent. That is non-stationarity again, this time from an adversary.

In [ ]:
show_video(evolution_frames(maddpg, k=5), fps=10)

## Same predators, different prey
The final predators against the frozen prey they first practised on, and against the final, trained prey.

In [ ]:
final, frozen_prey = load_policy(maddpg["ckpts"][-1]), snapshot(maddpg, 400_000)
vs_frozen = MixedPolicy([final, final, final, frozen_prey])       # agents: 3 predators, then the prey
vs_final = MixedPolicy([final] * 4)
for name, pol in (("vs frozen prey", vs_frozen), ("vs trained prey", vs_final)):
    print(f"{name:>16}: {evaluate(pol, 'tag', {}, 64)['contacts']:.2f} touch-steps / episode")
show_video(video(vs_frozen, "tag", {}, title="final predators vs frozen prey") +
           video(vs_final, "tag", {}, title="final predators vs trained prey"))

## Why do IDDPG and MADDPG look so similar here?
* In each run **both teams use the same algorithm**, so whatever MADDPG gives the predators, it gives the prey too.
* MADDPG's centralised critic sees the opponent's **actions**, which keeps its value estimates consistent. But it cannot foresee
  that the opponent will **change its strategy** next week. Chasing a moving target in a game is an *arms race*, not a value-estimation problem.
* The MADDPG paper shows its advantage by **pitting a MADDPG team against a DDPG team**.

What helps against an adapting opponent: training against **many past versions** of the opponent (self-play / populations),
**modelling the opponent**, and **looking ahead** before acting - the ideas behind AlphaZero, later in the workshop.